# MecâniQA — OAT 3 | Tunando o Meu Modelo

**Time:** Boa Vista — Alex Oliveira Santos, Alice Gomes Aragão, Ana Clara Ribeiro da Silva, Lorran Edley Matos Ribeiro, Rodrigo dos Anjos Lamartine

## Encontro 1: modelos com hiperparâmetros padrão

## 0. Preparação: importações, dados limpos e features (OAT 1 e 2)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_val_score
from IPython.display import Markdown, display
import time

SEED = 42
ALVOS = ["Trocas_Oleo", "Manutencao_Motor"]
NOMES = {"Trocas_Oleo": "Trocas de Óleo", "Manutencao_Motor": "Manutenção de Motor"}
pd.options.display.float_format = "{:.2f}".format

# dados e limpeza (OAT 1)
df = pd.read_excel("mecaniqa_dataset.xlsx")
df["Data"] = pd.to_datetime(df["Data"])
df = df.set_index("Data").sort_index().asfreq("D")

for alvo in ALVOS:
    df.loc[df[alvo] < 0, alvo] = np.nan
    df[alvo] = df[alvo].interpolate(method="time")
    q1, q3 = df[alvo].quantile([0.25, 0.75])
    iqr = q3 - q1
    df[alvo] = df[alvo].clip(lower=max(0, q1 - 1.5 * iqr), upper=q3 + 1.5 * iqr)

# features (OAT 2)
df_feat = df.copy()
for alvo in ALVOS:
    for lag in [1, 7, 30]:
        df_feat[f"{alvo}_lag_{lag}"] = df_feat[alvo].shift(lag)
    df_feat[f"{alvo}_rolling_mean_7"] = df_feat[alvo].rolling(7, min_periods=7).mean().shift(1)

df_feat = df_feat.dropna()
FEATURES = [c for c in df_feat.columns if c not in ALVOS]
print(f"Dataset de features: {df_feat.shape[0]} dias, {len(FEATURES)} features")

## 1. Divisão temporal (treino no passado, teste no futuro)

In [ ]:
n_treino = int(len(df_feat) * 0.80)
treino, teste = df_feat.iloc[:n_treino], df_feat.iloc[n_treino:]
X_treino, X_teste = treino[FEATURES], teste[FEATURES]

assert treino.index.max() < teste.index.min(), "O treino precisa terminar antes do teste!"
print(f"Treino: {treino.index.min().date()} a {treino.index.max().date()}  ({len(treino)} dias)")
print(f"Teste : {teste.index.min().date()} a {teste.index.max().date()}  ({len(teste)} dias)")

## 2. Baselines antigos (referência a ser batida) e função de erro

In [ ]:
def avaliar_metricas(y_real, y_previsto):
    y_real, y_prev = np.asarray(y_real, dtype=float), np.asarray(y_previsto, dtype=float)
    mascara = y_real != 0   # MAPE não é definido quando o valor real é zero
    return {
        "RMSE": np.sqrt(mean_squared_error(y_real, y_prev)),
        "MAPE": mean_absolute_percentage_error(y_real[mascara], y_prev[mascara]) * 100,
        "MAE": mean_absolute_error(y_real, y_prev),
    }


# baselines da OAT 2
def previsao_naive(serie):
    return serie.shift(1)

def previsao_media_movel(serie, janela=7):
    return serie.rolling(janela, min_periods=janela).mean().shift(1)


# baseline de cada variável = o de menor MAE no treino
baseline_prev, baseline_nome = {}, {}
for alvo in ALVOS:
    candidatos = {"Naive": previsao_naive(df[alvo]), "Média Móvel 7d": previsao_media_movel(df[alvo])}
    mae_treino = {n: avaliar_metricas(df[alvo].loc[treino.index], p.loc[treino.index])["MAE"] for n, p in candidatos.items()}
    baseline_nome[alvo] = min(mae_treino, key=mae_treino.get)
    baseline_prev[alvo] = candidatos[baseline_nome[alvo]].loc[teste.index]
    print(f"{NOMES[alvo]}: baseline antigo = {baseline_nome[alvo]}")

## 3. Passo 1 — Instanciar modelos clássicos de regressão (hiperparâmetros default)

In [ ]:
def criar_modelos():
    return {
        "Regressão Linear": LinearRegression(),
        "Árvore de Decisão": DecisionTreeRegressor(random_state=SEED),
        "Random Forest": RandomForestRegressor(random_state=SEED),
        "Gradient Boosting": GradientBoostingRegressor(random_state=SEED),
    }

modelos = {alvo: criar_modelos() for alvo in ALVOS}
print("Modelos instanciados:", list(modelos["Trocas_Oleo"].keys()))

## 4. Passo 2 e 3 — Pipeline: `fit()` só no treino e `predict()` no teste

In [ ]:
previsoes = {alvo: {} for alvo in ALVOS}

for alvo in ALVOS:
    for nome, modelo in modelos[alvo].items():
        modelo.fit(X_treino, treino[alvo])
        previsoes[alvo][nome] = modelo.predict(X_teste)

print("Treino e previsões concluídos para todos os modelos.")

## 5. Critério de aceite — erro inicial (RMSE e MAPE) dos modelos "crus" contra o baseline antigo

In [ ]:
for alvo in ALVOS:
    b = avaliar_metricas(teste[alvo], baseline_prev[alvo])
    print(f"\n=== {NOMES[alvo]} ===")
    print(f"Baseline antigo ({baseline_nome[alvo]}) - RMSE: {b['RMSE']:.2f}, MAPE: {b['MAPE']:.2f}%")
    for nome in modelos[alvo]:
        m = avaliar_metricas(teste[alvo], previsoes[alvo][nome])
        bate = "BATE o baseline" if (m["RMSE"] < b["RMSE"] and m["MAPE"] < b["MAPE"]) else "não bate o baseline"
        print(f"{nome} - RMSE: {m['RMSE']:.2f}, MAPE: {m['MAPE']:.2f}%  ->  {bate}")

### Tabela-resumo

In [ ]:
linhas = []
for alvo in ALVOS:
    b = avaliar_metricas(teste[alvo], baseline_prev[alvo])
    linhas.append({"Variável": NOMES[alvo], "Modelo": f"Baseline antigo ({baseline_nome[alvo]})", "RMSE": b["RMSE"], "MAPE (%)": b["MAPE"], "Bate o baseline?": "referência"})
    for nome in modelos[alvo]:
        m = avaliar_metricas(teste[alvo], previsoes[alvo][nome])
        bate = "SIM" if (m["RMSE"] < b["RMSE"] and m["MAPE"] < b["MAPE"]) else "não"
        linhas.append({"Variável": NOMES[alvo], "Modelo": nome, "RMSE": m["RMSE"], "MAPE (%)": m["MAPE"], "Bate o baseline?": bate})

pd.DataFrame(linhas).set_index(["Variável", "Modelo"])

---
## Encontro 2: Grid Search com TimeSeriesSplit

### 6.1 Qual é o nosso melhor modelo? (escolha feita só no treino, sem olhar o teste)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

candidatos = {
    "Árvore de Decisão": DecisionTreeRegressor(random_state=SEED),
    "Random Forest": RandomForestRegressor(random_state=SEED),
    "Gradient Boosting": GradientBoostingRegressor(random_state=SEED),
}

cv_inicial = {alvo: {} for alvo in ALVOS}
for alvo in ALVOS:
    for nome, modelo in candidatos.items():
        scores = cross_val_score(modelo, X_treino, treino[alvo], cv=tscv,
                                 scoring="neg_root_mean_squared_error", n_jobs=-1)
        cv_inicial[alvo][nome] = -scores.mean()

tabela_cv = pd.DataFrame(cv_inicial).rename(columns=NOMES)
tabela_cv.index.name = "RMSE na validação cruzada (treino)"
for alvo in ALVOS:
    print(f"{NOMES[alvo]:<20} menor RMSE na validação: {min(cv_inicial[alvo], key=cv_inicial[alvo].get)}")
tabela_cv

### 6.2 Passos 1 e 2 — Dicionário de hiperparâmetros e `GridSearchCV` com `TimeSeriesSplit`

In [ ]:
# hiperparâmetros
grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10, None],
    "min_samples_leaf": [1, 2, 4, 8],
}
n_combinacoes = int(np.prod([len(v) for v in grid.values()]))
print(f"Grid: {n_combinacoes} combinações x 5 folds = {n_combinacoes * 5} treinos por variável\n")

# GridSearchCV com TimeSeriesSplit
grid_busca = {}
for alvo in ALVOS:
    busca = GridSearchCV(
        estimator=RandomForestRegressor(random_state=SEED),
        param_grid=grid,
        cv=TimeSeriesSplit(n_splits=5),
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    inicio = time.perf_counter()
    busca.fit(X_treino, treino[alvo])
    duracao = time.perf_counter() - inicio
    grid_busca[alvo] = busca

    print(f"=== {NOMES[alvo]} ===")
    print("best_params_:", busca.best_params_)
    print(f"Melhor RMSE na validação cruzada: {-busca.best_score_:.3f}  | tempo: {duracao:.1f} s\n")

### 6.3 Critério de aceite — redução do erro em relação ao modelo da semana passada

In [ ]:
ganho = {}
for alvo in ALVOS:
    prev_inicial = previsoes[alvo]["Random Forest"]
    prev_tunado = grid_busca[alvo].best_estimator_.predict(X_teste)
    m_ini = avaliar_metricas(teste[alvo], prev_inicial)
    m_tun = avaliar_metricas(teste[alvo], prev_tunado)
    cv_ini, cv_tun = cv_inicial[alvo]["Random Forest"], -grid_busca[alvo].best_score_

    tabela = pd.DataFrame(
        {
            "RMSE validação cruzada (treino)": (cv_ini, cv_tun),
            "RMSE teste": (m_ini["RMSE"], m_tun["RMSE"]),
            "MAPE teste (%)": (m_ini["MAPE"], m_tun["MAPE"]),
            "MAE teste": (m_ini["MAE"], m_tun["MAE"]),
        },
        index=["RF inicial (default)", "RF tunado (Grid Search)"],
    ).T
    tabela["Redução do erro (%)"] = (tabela.iloc[:, 0] - tabela.iloc[:, 1]) / tabela.iloc[:, 0] * 100
    ganho[alvo] = tabela

    print(f"\n=== {NOMES[alvo]} ===")
    print("best_params_:", grid_busca[alvo].best_params_)
    print(tabela.to_string())

### Conclusão

In [ ]:
def frase_variacao(p):
    return f"redução de {p:.1f}%" if p >= 0 else f"aumento de {abs(p):.1f}%"


partes = ["### Resumo dos encontros\n"]
for alvo in ALVOS:
    b = avaliar_metricas(teste[alvo], baseline_prev[alvo])
    g = ganho[alvo]
    ini_rmse, tun_rmse = g.loc["RMSE teste"].iloc[0], g.loc["RMSE teste"].iloc[1]
    partes.append(
        f"**{NOMES[alvo]}**\n"
        f"- Encontro 1: o Random Forest sem ajustes já reduziu o RMSE do baseline de **{b['RMSE']:.2f}** para **{ini_rmse:.2f}**.\n"
        f"- Encontro 2: o Grid Search escolheu `{grid_busca[alvo].best_params_}`. No erro de validação houve "
        f"**{frase_variacao(g.loc['RMSE validação cruzada (treino)', 'Redução do erro (%)'])}**; no teste, o RMSE foi de {ini_rmse:.2f} para "
        f"**{tun_rmse:.2f}** ({frase_variacao(g.loc['RMSE teste', 'Redução do erro (%)'])}) e o MAPE teve "
        f"{frase_variacao(g.loc['MAPE teste (%)', 'Redução do erro (%)'])}.\n"
    )

ganhos_teste = [ganho[a].loc["RMSE teste", "Redução do erro (%)"] for a in ALVOS]
if all(abs(x) < 5 for x in ganhos_teste):
    partes.append(
        "**Leitura:** o tuning melhora o erro de validação (que é o que a busca otimiza), mas no teste o ganho adicional é pequeno, "
        "sinal de que o Random Forest com hiperparâmetros default já estava perto do teto para estes atributos. "
        "O maior salto de desempenho veio de trocar o baseline por um modelo de Machine Learning."
    )
else:
    partes.append("**Leitura:** o tuning trouxe ganho relevante também no teste, confirmando que vale a pena sintonizar o modelo.")
display(Markdown("\n".join(partes)))